# 04. Rede Neural Artificial (MLP) em PyTorch — Problema de Classificação Binária

Este notebook implementa, treina e analisa a Rede Neural Perceptron Multicamadas (`MLP`) em PyTorch para o problema de **Classificação Binária** (`late_delivery`: $0 = \text{No Prazo}$, $1 = \text{Atrasado}$), atendendo às Seções 1, 2 e 5.1 da Atividade Prática:
1. **Divisão Estratificada e Padronização sem *Data Leakage*:** Mesma divisão 80%/20% (`random_state=42`, `stratify=y`) das unidades anteriores e separação de 20% do treino como validação estratificada, com `StandardScaler` ajustado exclusivamente no treino.
2. **Justificativa da Saída em *Logit* e `nn.BCEWithLogitsLoss`:** 1 neurônio de saída linear (sem `Sigmoid` embutida na arquitetura), aplicando `torch.sigmoid` apenas na inferência e no cálculo de ROC-AUC e PR-AUC.
3. **Efeito do `pos_weight` e Otimização do Limiar de Decisão na Validação:** Análise do peso da classe positiva ($w_{+} = N_{\text{neg}} / N_{\text{pos}} \approx 1{,}734$) e escolha do limiar de decisão ótimo sobre o conjunto de validação.
4. **Treinamento Multi-Sementes (5 Sementes):** Comparação entre a configuração *Baseline* e a *Configuração Final Otimizada* com 5 sementes (`[42, 100, 2026, 7, 99]`).

## 1. Carregamento Estratificado, Padronização e Cálculo de `pos_weight`

In [ ]:
from IPython.display import display
import os
import sys
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.append("..")
from src.models import MLP, count_trainable_parameters, estimate_macs_per_sample
from src.train import (
    set_seeds,
    prepare_classification_data,
    predict_classification,
    evaluate_config_seeds,
)
from src.metrics import compute_classification_metrics, evaluate_thresholds_on_validation
from src.inference import salvar_checkpoint_completo
from src.visualization import plot_learning_curves_pair

sns.set_theme(style="whitegrid", palette="muted")
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)

data_clf = prepare_classification_data("../data/processed/dataset_classification.csv", test_size=0.20, val_size=0.20, random_state=42)

print(f"Número de features de entrada (in_dim): {data_clf['in_dim']}")
print(f"Sub-treino (64% total): {data_clf['X_tr'].shape[0]:,} amostras (Taxa atraso = {data_clf['y_tr'].mean()*100:.2f}%)")
print(f"Validação  (16% total): {data_clf['X_val'].shape[0]:,} amostras (Taxa atraso = {data_clf['y_val'].mean()*100:.2f}%)")
print(f"Teste      (20% total): {data_clf['X_test'].shape[0]:,} amostras (Taxa atraso = {data_clf['y_test'].mean()*100:.2f}%)")
print(f"Fator de desbalanceamento pos_weight (N_neg / N_pos no treino): {data_clf['pos_weight']:.4f}")

## 2. Justificativa Técnica: 1 Neurônio de Saída (*Logit*) + `nn.BCEWithLogitsLoss`

Para classificação binária em PyTorch, projetamos a camada de saída com **1 único neurônio linear sem função de ativação** ($z \in \mathbb{R}$, denominado *logit*) e utilizamos `nn.BCEWithLogitsLoss`:
1. **Estabilidade Numérica (*Log-Sum-Exp Trick*):** Se aplicássemos `torch.sigmoid(z)` na saída da rede seguida de `nn.BCELoss`, valores extremos de $z$ (ex.: $z < -20$ ou $z > 20$) saturariam a sigmoide em $0{,}0$ ou $1{,}0$ em precisão simples (`float32`), fazendo com que $\log(\sigma(z))$ produzisse `-inf` ou `NaN` durante o *backpropagation*. A `nn.BCEWithLogitsLoss` funde as duas operações na identidade analiticamente estável:
$$\ell(z, y) = \max(z, 0) - z \cdot y + \log\!\left(1 + e^{-|z|}\right)$$
2. **Probabilidades na Inferência:** Durante a avaliação e na construção das curvas ROC e Precision-Recall, convertemos os *logits* em probabilidades calibradas $\hat{p} = \sigma(z) = \frac{1}{1 + e^{-z}}$ usando `torch.sigmoid` dentro de `torch.inference_mode()`.

## 3. Treinamento Multi-Sementes (5 Sementes): *Baseline* vs. *Configuração Final* e Efeito do `pos_weight`

In [ ]:
SEEDS_5 = (42, 100, 2026, 7, 99)

cfg_baseline_clf = {
    "experimento": "comparacao_clf",
    "valor_testado": "Baseline (32, 32) ReLU Adam (BCE Padrão)",
    "hidden": (32, 32),
    "activation": "ReLU",
    "optimizer": "Adam",
    "lr": 1e-3,
    "batch_size": 256,
    "dropout": 0.0,
    "weight_decay": 0.0,
    "loss_name": "BCE",
    "n_epocas": 100,
    "paciencia": 15,
}

cfg_posweight_clf = {
    "experimento": "comparacao_clf",
    "valor_testado": "Baseline (32, 32) ReLU Adam (BCE com pos_weight=1.73)",
    "hidden": (32, 32),
    "activation": "ReLU",
    "optimizer": "Adam",
    "lr": 1e-3,
    "batch_size": 256,
    "dropout": 0.0,
    "weight_decay": 0.0,
    "loss_name": "BCE_pos_weight",
    "n_epocas": 100,
    "paciencia": 15,
}

cfg_final_clf = {
    "experimento": "comparacao_clf",
    "valor_testado": "Final (64, 64) GELU AdamW",
    "hidden": (64, 64),
    "activation": "GELU",
    "optimizer": "AdamW",
    "lr": 1e-3,
    "batch_size": 128,
    "dropout": 0.10,
    "weight_decay": 1e-4,
    "loss_name": "BCE",
    "n_epocas": 100,
    "paciencia": 15,
}

sum_base_clf, rows_base_clf, info_base_clf = evaluate_config_seeds("classification", data_clf, cfg_baseline_clf, seeds=SEEDS_5)
sum_pw_clf, rows_pw_clf, info_pw_clf = evaluate_config_seeds("classification", data_clf, cfg_posweight_clf, seeds=SEEDS_5)
sum_final_clf, rows_final_clf, info_final_clf = evaluate_config_seeds("classification", data_clf, cfg_final_clf, seeds=SEEDS_5)

df_resumo_clf = pd.DataFrame([sum_base_clf, sum_pw_clf, sum_final_clf])
df_resumo_clf.to_csv("../reports/tables/mlp_classificacao_baseline_vs_final_val.csv", index=False)
print("=== RESUMO AGREGADO (MÉDIA ± DESVIO PADRÃO ENTRE 5 SEMENTES) — CLASSIFICAÇÃO ===")
display(df_resumo_clf)

## 4. Curvas de Aprendizado e Otimização do Limiar de Decisão na Validação

Em problemas com classes moderadamente desbalanceadas ($63{,}43\%$ classe 0 vs. $36{,}57\%$ classe 1), o limiar padrão $\tau = 0{,}50$ privilegia a precisão em detrimento da sensibilidade (*Recall*). Avaliamos sistematicamente $\tau \in [0{,}15,\; 0{,}85]$ **exclusivamente no conjunto de validação** para escolher o limiar ótimo que maximiza o F1-score sem tocar no conjunto de teste.

In [ ]:
plot_learning_curves_pair(
    info_base_clf["hist"],
    info_final_clf["hist"],
    task_title="MLP Classificação Binária (Atraso de Entrega)",
    loss_label="Perda BCEWithLogitsLoss",
    filename="mlp_clf_learning_curves.png",
)

best_model_clf = info_final_clf["best_model"]
base_model_clf = info_base_clf["best_model"]

probs_val_final, _ = predict_classification(best_model_clf, data_clf["X_val"], threshold=0.5)
limiar_otimo, df_thr = evaluate_thresholds_on_validation(data_clf["y_val"], probs_val_final)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(df_thr["limiar"], df_thr["Precision"], label="Precision (Validação)", color="#2a9d8f", lw=2.2)
ax.plot(df_thr["limiar"], df_thr["Recall"], label="Recall / Sensibilidade (Validação)", color="#e76f51", lw=2.2)
ax.plot(df_thr["limiar"], df_thr["F1-score"], label="F1-Score (Validação)", color="#1d3557", lw=2.8)
ax.plot(df_thr["limiar"], df_thr["Accuracy"], label="Accuracy (Validação)", color="#8d99ae", linestyle=":", lw=1.8)
ax.axvline(0.50, color="gray", linestyle="--", lw=1.5, label="Limiar Padrão (0.50)")
ax.axvline(limiar_otimo, color="#e63946", linestyle="--", lw=2.0, label=f"Limiar Ótimo Validação ({limiar_otimo:.2f})")
ax.set_title("Escolha do Limiar de Decisão no Conjunto de Validação (Trade-off Precision vs. Recall)", fontsize=12, fontweight="bold")
ax.set_xlabel("Limiar de Decisão sobre a Probabilidade P(y=1|x)")
ax.set_ylabel("Valor da Métrica")
ax.legend()
plt.tight_layout()
plt.savefig("../reports/figures/mlp_clf_threshold_analysis.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"Limiar ótimo selecionado na Validação: {limiar_otimo:.3f}")

## 5. Avaliação Final no Teste e Salvamento do Checkpoint `models/modelo_classificacao.pt`

In [ ]:
prob_test_base, pred_test_base = predict_classification(base_model_clf, data_clf["X_test"], threshold=0.50)
prob_tr_final, pred_tr_final = predict_classification(best_model_clf, data_clf["X_tr"], threshold=limiar_otimo)
prob_val_final, pred_val_final = predict_classification(best_model_clf, data_clf["X_val"], threshold=limiar_otimo)
prob_test_final_05, pred_test_final_05 = predict_classification(best_model_clf, data_clf["X_test"], threshold=0.50)
prob_test_final_opt, pred_test_final_opt = predict_classification(best_model_clf, data_clf["X_test"], threshold=limiar_otimo)

m_base_te = compute_classification_metrics(data_clf["y_test"], pred_test_base, prob_test_base)
m_final_tr = compute_classification_metrics(data_clf["y_tr"], pred_tr_final, prob_tr_final)
m_final_val = compute_classification_metrics(data_clf["y_val"], pred_val_final, prob_val_final)
m_final_te_05 = compute_classification_metrics(data_clf["y_test"], pred_test_final_05, prob_test_final_05)
m_final_te_opt = compute_classification_metrics(data_clf["y_test"], pred_test_final_opt, prob_test_final_opt)

df_eval_clf = pd.DataFrame([
    {"Configuracao": "MLP Baseline (Limiar 0.50)", "Particao": "Teste (20%)", **{k: v for k, v in m_base_te.items() if k != "Confusion Matrix"}},
    {"Configuracao": f"MLP Final (Limiar 0.50)", "Particao": "Teste (20%)", **{k: v for k, v in m_final_te_05.items() if k != "Confusion Matrix"}},
    {"Configuracao": f"MLP Final (Limiar Val = {limiar_otimo:.2f})", "Particao": "Treino (64%)", **{k: v for k, v in m_final_tr.items() if k != "Confusion Matrix"}},
    {"Configuracao": f"MLP Final (Limiar Val = {limiar_otimo:.2f})", "Particao": "Validação (16%)", **{k: v for k, v in m_final_val.items() if k != "Confusion Matrix"}},
    {"Configuracao": f"MLP Final (Limiar Val = {limiar_otimo:.2f})", "Particao": "Teste (20%)", **{k: v for k, v in m_final_te_opt.items() if k != "Confusion Matrix"}},
])
display(df_eval_clf)

ckpt_clf = salvar_checkpoint_completo(
    caminho_pt="../models/modelo_classificacao.pt",
    modelo=best_model_clf,
    scaler=data_clf["scaler"],
    colunas_features=data_clf["feature_cols"],
    task="classification",
    limiar=limiar_otimo,
    classes=["No Prazo (0)", "Atrasado (1)"],
    hiperparametros_treino=cfg_final_clf,
)
size_clf_kb = os.path.getsize("../models/modelo_classificacao.pt") / 1024.0
macs_clf = estimate_macs_per_sample(best_model_clf)
print(f"\nCheckpoint salvo com sucesso em '../models/modelo_classificacao.pt' ({size_clf_kb:.2f} KB) | Complexidade: {macs_clf}")